# Assignment 1 - What does pretraining learn?
### COMP 5970, Generative AI and Foundational Models

Date: 02/11/2026

**Instructor:** Dr. Sathyanarayanan Aakur

**Teaching Assistant:** Chenjia Li

**Student:** Hannah Schmucker (hes0075@auburn.edu)

Comments:
* Make sure to run the cells in order; I defined common functions and variables in the first two blocks, so a block may not run correctly if the block(s) before it haven't been run.
* I made the batch sizes and sequence lengths small because I used all the GPU runtime on Colab and it was taking forever to run on the cpu :( Feel free to make them larger, if you'd like.

In [20]:
import numpy as np

import os, re, urllib.request, random, math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = "tinyshakespeare.txt"

if not os.path.exists(path):
        urllib.request.urlretrieve(url,path)

with open(path, "r", encoding="utf-8") as f:
    text = f.read().lower()


def tokenize_words(s: str):
    s = re.sub(r"\s+", " ", s)
    return re.findall(r"[a-z]+|[0-9]+|[^\w\s]",s)

tokens = tokenize_words(text)[:200_000]
print("Tokens:", len(tokens), "Unique Tokens:", len(set(tokens)))
print(tokens[:100])

cpu
Tokens: 200000 Unique Tokens: 9914
['first', 'citizen', ':', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'all', ':', 'speak', ',', 'speak', '.', 'first', 'citizen', ':', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', '?', 'all', ':', 'resolved', '.', 'resolved', '.', 'first', 'citizen', ':', 'first', ',', 'you', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', '.', 'all', ':', 'we', 'know', "'", 't', ',', 'we', 'know', "'", 't', '.', 'first', 'citizen', ':', 'let', 'us', 'kill', 'him', ',', 'and', 'we', "'", 'll', 'have', 'corn', 'at', 'our', 'own', 'price', '.', 'is', "'", 't', 'a', 'verdict', '?', 'all', ':', 'no', 'more', 'talking', 'on', "'", 't']


In [21]:
# === Construct vocab ===

# Any token that doesn't appear at least "min_count" times will be filtered
min_count = 5
max_vocab = 20_000

# Counts times a token appears
counts = Counter(tokens)

# Drop the tokens that don't appear at least "min_count" times
vocab = [w for w,c in counts.items() if c >= min_count]

vocab.sort(key=lambda w: counts[w], reverse=True)
vocab = vocab[:max_vocab-4]

PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
vocab = [PAD, UNK, BOS, EOS] + vocab

stoi = {w:i for i,w in enumerate(vocab)}
# Inverse mapping
itos = {i:w for w, i in stoi.items()}
V = len(vocab)

ids = [stoi.get(w,stoi[UNK]) for w in tokens]
ids = torch.tensor(ids, dtype=torch.long)
# uncomment to print vocab size and the idx of added tokens
print("Vocab:", V, "\nPadding id:", stoi[PAD], "\nUNK id:", stoi[UNK], "\nBOS id:", stoi[BOS], "\nEOS id:", stoi[EOS])

batch_size = 64
seq_len = 8

# === Define Batch ===
split = int(0.9 * len(ids))
train_ids = ids[:split]
test_ids = ids[split:]

# === Define steps, ema, alpha ===
steps = 1000
print_every = 250
ema = None
alpha = 0.98


Vocab: 2680 
Padding id: 0 
UNK id: 1 
BOS id: 2 
EOS id: 3


In [22]:
# ==== Unidirectional LM: train, test, evaluate loss ====
def get_unlm_batch(data_ids: torch.Tensor, batch_size=64, seq_len=8):
  """
  Returns:
  x         : [B,T] original
  y         : [B,T] next-token labels for forward LM
  """

  L = len(data_ids)
  max_start = L - (seq_len + 1)
  assert max_start > 0, "Oop, data too small for the chosen sequence length."

  starts = torch.randint(0,max_start,(batch_size,))
  x = torch.stack([data_ids[int(s.item()): int(s.item()) + seq_len] for s in starts])
  y_next = torch.stack([data_ids[int(s.item()) + 1: int(s.item()) + 1 + seq_len] for s in starts])

  return x.to(device), y_next.to(device)


# === Evalute loss ===
@torch.no_grad()
def eval_unlm_loss(model, data_ids, iters=100, batch_size=64, seq_len=8):
  model.eval()
  losses = []
  for _ in range(iters):
      x, y_next = get_unlm_batch(data_ids,batch_size=batch_size, seq_len=seq_len)

      # forward LM on x : [B,T,V]
      logits_fwd, _ = model(x)

      loss_f = F.cross_entropy(logits_fwd.reshape(-1,V), y_next.reshape(-1))
      losses.append(loss_f.item())

  model.train()
  return sum(losses) / len(losses)


# === Define forward LSTM ===
class UnidirectionalLM(nn.Module):

  """
  - forward LSTM predicts the next token at each position
  - contextual embedding at position t: concat(h_fwd[t])
  """

  def __init__(self, vocab_size, emb_dim=32, hidden_dim=64,num_layers=1):
    super().__init__()

    # Input Embedding Table for UNLM = [vocab_size, emb_dim]
    self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD])

    self.fwd = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
    self.out_fwd = nn.Linear(hidden_dim, vocab_size)

  def forward(self,x):
      # embed the tokens
      e = self.emb(x)                       # shape = [B,T,D]

      # run forward LM
      h_fwd, _ = self.fwd(e)                # [B,T,H]
      logits_fwd = self.out_fwd(h_fwd)      # [B,T,V]

      # contextual embedding
      uni_ctx = h_fwd                           # [B,T,H]

      # returns next-token predictions
      return logits_fwd, uni_ctx


model = UnidirectionalLM(V, emb_dim=32, hidden_dim=64).to(device)
unidir_model = model
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

print(f"Model size: {sum(p.numel() for p in model.parameters())/1e6:.2f}")


# === Train ===
model.train()
for step in range(1, steps+1):
  x, y_next = get_unlm_batch(train_ids, batch_size, seq_len)
  # forward pass the input
  logits_fwd, _ = model(x)

  loss_f = F.cross_entropy(logits_fwd.reshape(-1, V), y_next.reshape(-1))

  # clear old gradients
  opt.zero_grad()
  # backpropagate
  loss_f.backward()

  # prevent exploding gradients
  nn.utils.clip_grad_norm_(model.parameters(),1.0)
  # update model weights
  opt.step()

  ema = loss_f.item() if ema is None else alpha*ema + (1-alpha)*loss_f.item()


# uncomment to print steps, training loss, test loss, and test perplexity
  #if step % print_every == 0:
    #te = eval_unlm_loss(model, test_ids, iters=200)
    #print(f"step {step:4d}/{steps} | traim_ema {ema:.3f} | test {te:.3f} | test_ppl~ {math.exp(te):.2f}")


Model size: 0.29


In [23]:
# ==== Bidirectional (ELMo-style) LM: train, test, evaluate loss ====
def get_bilm_batch(data_ids: torch.Tensor, batch_size=64, seq_len=8):
    """
    Returns:
      x      : [B,T] original
      y_next : [B,T] next-token labels for forward LM
      x_rev  : [B,T] reversed
      y_next_rev: [B,T] next-token labels for backward LM (on reversed)
    """
    L = len(data_ids)
    max_start = L - (seq_len + 1)
    assert max_start > 0, "Oops, data too small for the chosen sequence length."

    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data_ids[int(s.item()): int(s.item()) + seq_len] for s in starts])
    y_next = torch.stack([data_ids[int(s.item()) + 1: int(s.item()) + 1 + seq_len] for s in starts])

    x_rev = torch.flip(x, dims=[1])             # reverse
    y_next_rev = torch.flip(y_next, dims=[1])   # reverse labels

    return x.to(device), y_next.to(device), x_rev.to(device), y_next_rev.to(device)


# === Evalute loss ===
@torch.no_grad()
def eval_bilm_loss(model, data_ids, iters=100, batch_size=64, seq_len=8):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y_next, x_rev, y_next_rev = get_bilm_batch(data_ids, batch_size=batch_size, seq_len=seq_len)

        # forward LM
        logits_fwd, _, _ = model(x)

        # backward LM
        logits_bwd, _, _ = model(x_rev)

        loss_f = F.cross_entropy(logits_fwd.reshape(-1, V), y_next.reshape(-1))
        loss_b = F.cross_entropy(logits_bwd.reshape(-1, V), y_next_rev.reshape(-1))
        losses.append((loss_f + loss_b).item())

    model.train()
    return sum(losses) / len(losses)


# === Forward and Backward LSTM (ELMo-style) ===
class NaiveELMoBiLM(nn.Module):
    """
    - forward LSTM predicts next token at each position
    - backward LSTM predicts previous token at each position (trained on reversed input)
    - contextual embedding at position t: concat(h_fwd[t], h_bwd[t])
    """
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64, num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD])

        self.fwd = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.bwd = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)

        self.out_fwd = nn.Linear(hidden_dim, vocab_size)
        self.out_bwd = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: [B,T]
        e = self.emb(x)                               # [B,T,D]

        # forward LM
        h_fwd, _ = self.fwd(e)                        # [B,T,H]
        logits_fwd = self.out_fwd(h_fwd)              # [B,T,V]

        # backward LM
        e_rev = torch.flip(e, dims=[1])
        h_bwd_rev, _ = self.bwd(e_rev)                # [B,T,H] aligned to reversed time
        h_bwd = torch.flip(h_bwd_rev, dims=[1])       # [B,T,H] aligned to og positions
        logits_bwd = self.out_bwd(h_bwd)              # [B,T,V]

        # contextual embedding: [B,T,2H]
        bi_ctx = torch.cat([h_fwd, h_bwd], dim=-1)

        return logits_fwd, logits_bwd, bi_ctx

model = NaiveELMoBiLM(V, emb_dim=32, hidden_dim=64).to(device)
bidir_model = model
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

print(f"Model size: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

# === Train ===
"""
  Similiar to training the Unidirectional LSTM (except
    losses are the sum of fwd and bwd context embeddings).
"""
model.train()
for step in range(1, steps+1):
    x, y_next, x_rev, y_next_rev = get_bilm_batch(train_ids, batch_size, seq_len)
    logits_fwd, logits_bwd, _ = model(x)

    loss_f = F.cross_entropy(logits_fwd.reshape(-1, V), y_next.reshape(-1))
    loss_b = F.cross_entropy(logits_bwd.reshape(-1, V), y_next_rev.reshape(-1))
    loss = loss_f + loss_b

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    ema = loss.item() if ema is None else alpha*ema + (1-alpha)*loss.item()

# uncomment to print steps, training loss, test loss, and test perplexity
    #if step % print_every == 0:
      #te = eval_bilm_loss(model, test_ids, iters=200)
      #print(f"step {step:4d}/{steps} | train_ema {ema:.3f} | test {te:.3f} | test_ppl~ {math.exp(te):.2f}")


Model size: 0.48M parameters


In [25]:
# ==== Seq2Seq: train, test, evaluate loss ====

def make_reversal_batch(data_ids: torch.Tensor, batch_size=64,seq_len=6):
  """
    Unlike the other models, we'll cut the seq_len
      so that the reversal stays accurate.
  """

  # sample (random, contiguous) sequences
  max_start = len(data_ids) - seq_len
  starts = torch.randint(0,max_start, (batch_size,))
  src = torch.stack([data_ids[s:s+seq_len] for s in starts])
  bos = torch.full((batch_size, 1), stoi[BOS], dtype=torch.long)
  eos = torch.full((batch_size, 1), stoi[EOS], dtype=torch.long)
  tgt = torch.flip(src, dims=[1])                                        # [B, T]
  tgt_in  = torch.cat([bos, tgt], dim=1)                                 # [B, T+1]
  tgt_out = torch.cat([tgt, eos], dim=1)
  return src.to(device), tgt_in.to(device), tgt_out.to(device)

# === Seq2Seq model (encoder-decoder) ===
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD])
        self.encoder = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def encode(self, src):
        e = self.emb(src)
        encoder_out, (h, c) = self.encoder(e)
        return encoder_out,(h, c)

    def decode(self, tgt_in, state):
        e = self.emb(tgt_in)
        out, state = self.decoder(e, state)
        logits = self.fc(out)
        return logits, state

    def forward(self, src, tgt_in):
        encoder_out, state = self.encode(src)
        logits, _ = self.decode(tgt_in, state)
        return logits

model = Seq2Seq(V, emb_dim=32, hidden_dim=64).to(device)
seq2seq_model = model
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

# === Train ===
def eval_loss(iters=100):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(iters):
            src, tgt_in, tgt_out = make_reversal_batch(test_ids, batch_size=batch_size, seq_len=seq_len)
            logits = model(src, tgt_in)                        # [B, T+1, V]
            loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1))
            losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

for step in range(1, steps+1):
    src, tgt_in, tgt_out = make_reversal_batch(train_ids, batch_size=batch_size, seq_len=seq_len)
    logits = model(src, tgt_in)
    loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1))

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    ema = loss.item() if ema is None else alpha*ema + (1-alpha)*loss.item()

# uncomment to print steps, training loss, test loss, and test perplexity
    #if step % print_every == 0:
        #te = eval_loss(iters=100)
        #print(f"step {step:4d}/{steps} | train_ema {ema:.3f} | test {te:.3f} | test_ppl {math.exp(te):.2f}")


In [26]:
# ==== Eval 1: Static Embedding Neighbors ====

def cosine(a, b, eps=1e-8):
    a = a / (a.norm() + eps)
    b = b / (b.norm() + eps)
    return float((a*b).sum())

@torch.no_grad()
def static_embedding_neighbors(word, topk=8):
  if word not in stoi:
      print(f"{word} does not appear in the vocabulary.")
      return []

  embedding_layer = model.emb.weight    # [V,D]
  query = embedding_layer[stoi[word]]

  similarity = []
  for i in range(embedding_layer.shape[0]):
    similarity.append((i, cosine(query,embedding_layer[i])))

  similarity.sort(key=lambda x: x[1], reverse=True)

  out = []
  for i, s in similarity[:topk+1]:
    out.append((i,itos[i],s))

  return out

models = [unidir_model, bidir_model, seq2seq_model]
model_name = ["Unidirectional LSTM LM", "Bidirectional LSTM LM", "Seq2Seq LSTM LM"]

for m, n in zip(models, model_name):
  model = m
  print(f"\n===={n}====\n ")

  for word in ["love", "death", "man", "woman"]:
    print(f"\nNeighbors of '{word}':\n")
    for i,tok,similarity in static_embedding_neighbors(word,topk=8):
      print(f"{tok:10s} sim={similarity:.5f}")


====Unidirectional LSTM LM====
 

Neighbors of 'love':

love       sim=1.00000
strange    sim=0.56647
volscians  sim=0.56173
sorrow     sim=0.55962
lead       sim=0.53347
bereft     sim=0.52063
countenance sim=0.51934
misery     sim=0.51875
furnish    sim=0.50983

Neighbors of 'death':

death      sim=1.00000
indeed     sim=0.61520
wear       sim=0.57146
course     sim=0.54046
branches   sim=0.52802
intents    sim=0.49908
herself    sim=0.46729
ta         sim=0.45862
image      sim=0.45764

Neighbors of 'man':

man        sim=1.00000
palace     sim=0.54036
it         sim=0.52662
misery     sim=0.51984
month      sim=0.50662
record     sim=0.50434
body       sim=0.50203
sail       sim=0.49559
ship       sim=0.49332

Neighbors of 'woman':

woman      sim=1.00000
countenance sim=0.56322
laugh      sim=0.55821
utter      sim=0.50641
thought    sim=0.50257
hit        sim=0.49943
avoid      sim=0.49939
provoked   sim=0.49550
suck       sim=0.48684

====Bidirectional LSTM LM====
 

Neighbors

In [27]:
# ==== Eval 2: Contextual Word Similarity ====

# === Extract contextual embeddings ===
@torch.no_grad()
def uni_encode_sentence(tokens_list):
  x = torch.tensor([[stoi.get(w, stoi[UNK]) for w in tokens_list]], dtype=torch.long, device=device)
  _, uni_ctx =unidir_model(x)  # [1,T,H]
  return uni_ctx[0].detach().cpu(), x[0].detach().cpu()

def bi_encode_sentence(tokens_list):
  x = torch.tensor([[stoi.get(w, stoi[UNK]) for w in tokens_list]], dtype=torch.long, device=device)
  _, _, bi_ctx = bidir_model(x)  # [1,T,2H]
  return bi_ctx[0].detach().cpu(), x[0].detach().cpu()

def seq2seq_encode_sentence(tokens_list):
  x = torch.tensor([[stoi.get(w, stoi[UNK]) for w in tokens_list]], dtype=torch.long, device=device)
  encoder_out, _ = seq2seq_model.encode(x)  # [1,T,h_dim]
  return encoder_out[0].detach().cpu(), x[0].detach().cpu()

# Two contexts containing the same word "king"
s1 = tokenize_words("i polish the mirror .")[:seq_len]
s2 = tokenize_words("the nail polish is red .")[:seq_len]

# Get the contextual embeddings for the tokens in each sentence (and token IDs)
ctx1_uni, ids1_uni = uni_encode_sentence(s1)
ctx2_uni, ids2_uni = uni_encode_sentence(s2)

ctx1_bi, ids1_bi = bi_encode_sentence(s1)
ctx2_bi, ids2_bi = bi_encode_sentence(s2)

ctx1_s2s, ids1_s2s = seq2seq_encode_sentence(s1)
ctx2_s2s, ids2_s2s = seq2seq_encode_sentence(s2)

def find_positions(tokens_list, word):
  return [i for i,w in enumerate(tokens_list) if w == word]

# Find the position of 'polish' in each sentence
word = "polish"
p1 = find_positions(s1, word)
p2 = find_positions(s2, word)

if p1 and p2:
    sim_uni = cosine(ctx1_uni[p1[0]], ctx2_uni[p2[0]])
    sim_bi = cosine(ctx1_bi[p1[0]], ctx2_bi[p2[0]])
    sim_s2s = cosine(ctx1_s2s[p1[0]], ctx2_s2s[p2[0]])
    print(f"Cosine similarity of contextual '{word}' in two contexts, using the Unidirectional LSTM: {sim_uni:.4f}")
    print(f"Cosine similarity of contextual '{word}' in two contexts, using the Bidirectional, ELMo style LSTM: {sim_bi:.4f}")
    print(f"Cosine similarity of contextual '{word}' in two contexts, using the Seq2Seq LSTM: {sim_s2s:.4f}")
else:
    print(f"The word,'{word}', was not found in either of the sentences.")


Cosine similarity of contextual 'polish' in two contexts, using the Unidirectional LSTM: 0.8156
Cosine similarity of contextual 'polish' in two contexts, using the Bidirectional, ELMo style LSTM: 0.8839
Cosine similarity of contextual 'polish' in two contexts, using the Seq2Seq LSTM: 0.8030


**Q1: What is "the embedding" in your code?**

For Eval 1, the tensors are input embedding tables defined in the __init__ functions (dimensions are [vocab, emb_dim]). The cosine similarity in Eval 1 falls between 0.2-0.7 and since the embedding tables lack context, the words 'most similar' are somewhat random.

In Eval 2, the tensors are...

(a) Unidirectional LM = the hidden state

(b) Bidirectional LM = hidden states of the forward and backward LSTMs (concatentated)

(c) Seq2Seq = the output of the encoder



**Q2: Effect of bidirectionality**

The cosine similarity of "polish" changes between the unidirectional LM (0.8974) and the bidirectional LM (0.9240) due to the addition of context. As stated in class, the forward LM has the objective of predicting the next word, while the backwards LM tries to predict the previous word. The result is that a forward pass captures semantic meaning and the backward pass captures syntax. In Eval 2, this means that the backward LM 'notices' that polish is used as a verb in the first example, but as a part of a noun in the second.

**Q3: Why Seq2Seq behaves differently**

It behaves differently because the training objective is different. Rather than trying to predict next/previous words (i.e. unconditionally), Seq2Seq uses the encoder to produce hidden states with the *goal* of the decoder being able to put together a sequence (i.e it has a conditional training objective). This is why the Seq2Seq's highest cosine similarity in Eval 1 is 0.34 (besides the trivial value of 1) while the other LM have higher values of 0.71 [undirectional] and 0.59 [bidirectional].

**Q4: One intentional change**

*I cut the seq_len from 16 to 8 [and 12 to 6 for Seq2Seq]*

Results: unidirectional, bidirectional, seq2seq: 0.8974, 0.9240,0.7795 -> 0.8156, 0.8839, 0.8030

The unidirectional and bidirectional LM's experienced a drop in cosine similarity because their training objectives become more accurate the more familiar they become with a token, their cosine similarity dropped. Hence, smaller seq_length means lower similarity.

The Seq2Seq model experienced a rise in cosine similarity because the model focused less on the conditional objective (needed to reconstruct a sequence) and therefore the token ("polish") had more similarity when compared to itself.